In [1]:
import os
import sys

import numpy as np
import pandas as pd

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing
from data_visualizing import DataVisualizing

## Load Data

In [2]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)

### Load Ground Truth Data

In [10]:
ground_truth_path = os.path.join(base_data_path, "financial_phrase_bank/annotators/fpb-maya-binary-imbalanced-96d-v1.csv")
print(ground_truth_path)
ground_truth_df = DataProcessing.load_from_file(ground_truth_path)
ground_truth_df

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/financial_phrase_bank/annotators/fpb-maya-binary-imbalanced-96d-v1.csv


,Base Sentence,maya_label,Sentence Label,Author Type
0,With the new production plant the company woul...,PREDICTION,1,1
1,According to the company 's updated strategy f...,PREDICTION,1,1
2,TeliaSonera TLSN said the offer is in line wit...,PREDICTION,1,1
3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMEN...",PREDICTION,1,1
4,The company also estimates the already carried...,PREDICTION,1,1
...,...,...,...,...
4841,LONDON MarketWatch -- Share prices ended lower...,NON-PREDICTION,0,1
4842,Rinkuskiai 's beer sales fell by 6.5 per cent ...,NON-PREDICTION,0,1
4843,Operating profit fell to EUR 35.4 mn from EUR ...,NON-PREDICTION,0,1
4844,Net sales of the Paper segment decreased to EU...,NON-PREDICTION,0,1


### Load Test Data

In [4]:
test_data_name = "cnn"

if test_data_name == "cnn":
    cnn_path = os.path.join(base_data_path, "classification_results/fpb-maya-binary-imbalanced-96d-v1_2026-03-14/cnn_prediction_results.csv")
    df = DataProcessing.load_from_file(cnn_path)
    filt_full = (df['prediction_type'] == "full")
    full_df = df[filt_full]
    full_df['Sentence Label'] = 1
    not_full_df = df[~filt_full]
    not_full_df['Sentence Label'] = 0
    test_df = DataProcessing.concat_dfs([full_df, not_full_df])

elif test_data_name == "all":
    pass
else:
    print("No correct test dataset name")


test_df

/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_75464/771201322.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  full_df['Sentence Label'] = 1
/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_75464/771201322.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  not_full_df['Sentence Label'] = 0


,text,prediction_type,source,target,date,outcome,components_found,confidence,Sentence Label
0,"However , Biohit estimates its total net sales...",full,Biohit estimates,net sales,2009,net sales,4,1.00,1
1,Sanoma Magazines ' net sales are estimated to ...,full,Sanoma Magazines ',net,2006,sales,4,1.00,1
2,Metso expects its net sales to increase by abo...,full,Metso expects its,about,2008,net sales,4,1.00,1
3,Cencorp estimates that its net sales in the la...,full,its net sales,EUR4,.0 m,last quarter will,4,1.00,1
4,The acquisition is expected to improve access ...,full,chrome ore resources,The,Turkey,acquisition,4,1.00,1
...,...,...,...,...,...,...,...,...,...
4841,LONDON MarketWatch -- Share prices ended lower...,partial,MarketWatch -- Share prices ended,London,NaN,offset broader weakness,3,0.75,0
4842,Rinkuskiai 's beer sales fell by 6.5 per cent ...,partial,Rinkuskiai 's,6.5,NaN,per cent,3,0.75,0
4843,Operating profit fell to EUR 35.4 mn from EUR ...,partial,EUR 35.4 mn,NaN,from EUR,mn,3,0.75,0
4844,Net sales of the Paper segment decreased to EU...,partial,Net sales of,NaN,mn,mn,3,0.75,0


## Metrics

In [5]:
y = ground_truth_df['Sentence Label'].values
y_hat = test_df['Sentence Label'].values

In [ ]:
EvaluationMetric.eval_classification_report(y, y_hat)

              precision    recall  f1-score   support

           0       1.00      0.97      0.99      4388
           1       0.78      1.00      0.88       458

    accuracy                           0.97      4846
   macro avg       0.89      0.99      0.93      4846
weighted avg       0.98      0.97      0.98      4846



{'0': {'precision': 1.0,
  'recall': 0.9708295350957156,
  'f1-score': 0.9851988899167438,
  'support': 4388.0},
 '1': {'precision': 0.7815699658703071,
  'recall': 1.0,
  'f1-score': 0.8773946360153256,
  'support': 458.0},
 'accuracy': 0.9735864630623194,
 'macro avg': {'precision': 0.8907849829351535,
  'recall': 0.9854147675478577,
  'f1-score': 0.9312967629660347,
  'support': 4846.0},
 'weighted avg': {'precision': 0.979355972837103,
  'recall': 0.9735864630623194,
  'f1-score': 0.9750102088835516,
  'support': 4846.0}}

In [7]:
EvaluationMetric.get_confusion_matrix(y_true=y, y_prediction=y_hat)

array([[4260,  128],
       [   0,  458]])

In [ ]:
EvaluationMetric.get_roc_auc(y_true=y, y_prediction=y_hat)
DataVisualizing.roc_curve()

np.float64(0.9854147675478577)

In [9]:
EvaluationMetric.get_pr_auc(y_true=y, y_prediction=y_hat)

np.float64(0.7815699658703071)